# AdaBoost from scratch
Three stages of AdaBoost on a 10-row toy dataset, written out by hand: weights, a decision stump, the error, alpha,
the weight update, normalisation and upsampling. Then the final weighted vote, and a check against scikit-learn.
The figures of the Note are drawn by `images/figs.py` with the same data and seed.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier, export_text

## 1. The data: 10 rows, two inputs, classes 1 and 0

In [ ]:
df = pd.DataFrame()
df["X1"] = [1, 2, 3, 4, 5, 6, 6, 7, 9, 9]
df["X2"] = [5, 3, 6, 8, 1, 9, 5, 8, 9, 2]
df["label"] = [1, 1, 0, 1, 0, 1, 0, 1, 0, 0]
df

In [ ]:
# a scatter plot: the two classes are mixed, no single straight cut separates them
px.scatter(df, x="X1", y="X2", color=df["label"].astype(str), width=500, height=400)

## 2. Stage 1

In [ ]:
# step 1: every row starts with weight 1/n = 0.1
df["weights"] = 1 / df.shape[0]

# step 2: a decision stump is a tree with max_depth=1
# (random_state=0 fixes which cut wins when two cuts are equally good)
X = df[["X1", "X2"]].values
y = df["label"].values
dt1 = DecisionTreeClassifier(max_depth=1, random_state=0)
dt1.fit(X, y)
print(export_text(dt1, feature_names=["X1", "X2"]))

In [ ]:
# its prediction for every training row
df["y_pred"] = dt1.predict(X)
df

In [ ]:
# step 3: the error is the total weight of the misclassified rows
def model_error(d):
    wrong = d["label"] != d["y_pred"]
    return d.loc[wrong, "weights"].sum()


# step 4: alpha = 1/2 ln((1 - error) / error)
# a tiny number keeps the division safe if the error is exactly 0
def calculate_model_weight(error, eps=1e-10):
    return 0.5 * np.log((1 - error + eps) / (error + eps))


error1 = model_error(df)
alpha1 = calculate_model_weight(error1)
print(f"error = {error1:.2f}, alpha1 = {alpha1:.4f}")

In [ ]:
# step 5: misclassified rows x e^alpha, correct rows x e^(-alpha)
def update_row_weights(d, alpha):
    correct = d["label"] == d["y_pred"]
    return np.where(correct, d["weights"] * np.exp(-alpha),
                    d["weights"] * np.exp(alpha))


df["updated_weights"] = update_row_weights(df, alpha1)
print("sum of the updated weights:", round(df["updated_weights"].sum(), 4))

In [ ]:
# step 6: normalise, so the weights add up to 1 again
df["normalized_weights"] = df["updated_weights"] / df["updated_weights"].sum()

# the ranges for upsampling: each row owns [lower, upper) on the line from 0 to 1
df["cumsum_upper"] = np.cumsum(df["normalized_weights"])
df["cumsum_lower"] = df["cumsum_upper"] - df["normalized_weights"]
df.round(4)

In [ ]:
# step 7: upsampling. Draw n random numbers between 0 and 1;
# each one picks the row whose range it falls in
rng = np.random.default_rng(0)


def create_new_dataset(d):
    indices = []
    for _ in range(d.shape[0]):
        a = rng.random()
        # the first row whose upper end is above a
        indices.append(int(np.argmax(d["cumsum_upper"].values > a)))
    return indices


index_values = create_new_dataset(df)
index_values

In [ ]:
# the new dataset: those rows, in that order; a fresh 0..9 index
# (the column 'row' remembers which original row each one is)
second_df = df.iloc[index_values][["X1", "X2", "label"]].reset_index(names="row")
second_df

## 3. Stage 2: the same steps on the new dataset

In [ ]:
# weights start again at 1/n: the upsampling has already done the reweighting
second_df["weights"] = 1 / second_df.shape[0]
X = second_df[["X1", "X2"]].values
y = second_df["label"].values
dt2 = DecisionTreeClassifier(max_depth=1, random_state=0).fit(X, y)
print(export_text(dt2, feature_names=["X1", "X2"]))
second_df["y_pred"] = dt2.predict(X)

error2 = model_error(second_df)
alpha2 = calculate_model_weight(error2)
print(f"error = {error2:.2f}, alpha2 = {alpha2:.4f}")
second_df

In [ ]:
second_df["updated_weights"] = update_row_weights(second_df, alpha2)
second_df["normalized_weights"] = (second_df["updated_weights"]
                                   / second_df["updated_weights"].sum())
second_df["cumsum_upper"] = np.cumsum(second_df["normalized_weights"])
second_df["cumsum_lower"] = second_df["cumsum_upper"] - second_df["normalized_weights"]
second_df.round(4)

In [ ]:
index_values = create_new_dataset(second_df)
third_df = (second_df.iloc[index_values][["row", "X1", "X2", "label"]]
            .reset_index(drop=True))
third_df

## 4. Stage 3: trained on the third dataset

In [ ]:
third_df["weights"] = 1 / third_df.shape[0]
X = third_df[["X1", "X2"]].values
y = third_df["label"].values
dt3 = DecisionTreeClassifier(max_depth=1, random_state=0).fit(X, y)
print(export_text(dt3, feature_names=["X1", "X2"]))
third_df["y_pred"] = dt3.predict(X)

error3 = model_error(third_df)
alpha3 = calculate_model_weight(error3)
print(f"error = {error3:.2f}, alpha3 = {alpha3:.4f}")
third_df

In [ ]:
print("alphas:", round(alpha1, 4), round(alpha2, 4), round(alpha3, 4))

## 5. Prediction: the weighted vote

In [ ]:
# a stump predicts 1 or 0; the vote needs +1 or -1
def to_pm(p):
    return 2 * p - 1


def adaboost_predict(query):
    query = np.array(query).reshape(1, 2)
    votes = [to_pm(m.predict(query)[0]) for m in (dt1, dt2, dt3)]
    total = alpha1 * votes[0] + alpha2 * votes[1] + alpha3 * votes[2]
    return votes, total, np.sign(total)


for q in ([1, 5], [9, 9]):
    votes, total, sign = adaboost_predict(q)
    print(q, "votes", votes, "total", round(total, 4), "sign", sign)

In [ ]:
# the vote on all 10 training rows
pred = [adaboost_predict(row)[2] for row in df[["X1", "X2"]].values]
truth = to_pm(df["label"].values)
print("predicted:", np.array(pred, dtype=int))
print("true:     ", truth)
print("training accuracy:", np.mean(np.array(pred) == truth))

## 6. Extra: weights instead of upsampling, and scikit-learn
scikit-learn does not draw new datasets. It passes the weights to the stump with `sample_weight`, and its alpha
(SAMME) has no factor 1/2. The loop below does the same; with the same tie-breaking, the alphas and the predictions match `AdaBoostClassifier`.

In [ ]:
def adaboost_weights(X, y, seeds):
    w = np.full(len(y), 1 / len(y))
    stumps, alphas = [], []
    for seed in seeds:
        stump = DecisionTreeClassifier(max_depth=1, random_state=seed)
        stump.fit(X, y, sample_weight=w)
        wrong = stump.predict(X) != y
        err = w[wrong].sum()
        alpha = np.log((1 - err) / err)      # SAMME: no 1/2
        w = w * np.exp(alpha * wrong)        # only mistakes grow
        w = w / w.sum()
        stumps.append(stump)
        alphas.append(alpha)
    return stumps, np.array(alphas)


X, y = df[["X1", "X2"]].values, df["label"].values
abc = AdaBoostClassifier(DecisionTreeClassifier(max_depth=1),
                         n_estimators=3, random_state=0).fit(X, y)

# scikit-learn gives each stump its own random_state; we reuse them,
# so that ties between equally good cuts are broken the same way
seeds = [s.random_state for s in abc.estimators_]
stumps, alphas = adaboost_weights(X, y, seeds)
ours = np.sign(sum(a * to_pm(s.predict(X)) for a, s in zip(alphas, stumps)))
print("our alphas:    ", alphas.round(4))
print("sklearn alphas:", abc.estimator_weights_.round(4))
print("our predictions:    ", ((ours + 1) // 2).astype(int))
print("sklearn predictions:", abc.predict(X))